# Instructor dry run: will students see a real change after fine-tuning?

Run this **once, a few days before class**. For each of the five scenarios it measures, on **100** held-out test cases:

* the two baselines: **prompt only** and **prompt + policy docs**,
* **fine-tuned** accuracy at different amounts of clean and noisy training data (the student settings cards),
* how long training takes on your GPU.

It then tells you, per scenario, the **fewest training examples that give a clear improvement**, meaning the fine-tuned 95% range sits entirely above the baseline's range. Use that to set the default `N_EXAMPLES` and the settings cards.

| `MODE` | Fine-tuning runs per scenario | Rough time, 5 scenarios |
|---|---|---|
| `"quick"` | 100 clean, 100 noisy (cards B and D) | ~1 hour on a free T4 |
| `"full"` | 25 / 50 / 100 / 200 / 300 clean, 100 / 300 noisy | ~3 hours on a T4, ~1 hour on an L4 or A100 (Colab Pro) |

*Times are estimates. The report prints the measured speed.* Results are saved after every run, so if Colab disconnects, re-run the cells and it continues where it stopped.

## 1. Setup

Runtime → Change runtime type → **T4 GPU** (or L4 / A100 with Colab Pro), then run:

In [ ]:
# Downloads the lab materials and installs two libraries (~1 minute)
REPO = "https://github.com/manaspandit27-web/manas.git"
BRANCH = "claude/nemotron-finetune-lab"
import os, sys
if not os.path.exists("/content/lab-repo"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/lab-repo
os.chdir("/content/lab-repo/nemotron-lab")
!pip install -q bitsandbytes peft
sys.path.insert(0, ".")
from lab import core
from IPython.display import Markdown, display
import pandas as pd
print("Ready. Scenarios:")
display(pd.DataFrame(core.list_scenarios()))

In [ ]:
# Options
MODE = "quick"          # "quick" or "full"
SCENARIOS = [s["scenario"] for s in core.list_scenarios()]   # or e.g. ["invoice_intake", "expense_audit"]
SAVE_TO_DRIVE = True    # keeps results if the Colab session ends (asks for Google Drive permission)

OUT = "dry_run_results.csv"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/nemotron_dry_run_results.csv"
SETTINGS = core.QUICK_SWEEP if MODE == "quick" else core.FULL_SWEEP
print("Results file:", OUT)
print("Fine-tuning settings (examples, data, epochs):", SETTINGS)

In [ ]:
# Loads NVIDIA Nemotron onto the GPU (~3 minutes)
model, tok = core.load_model()

## 2. Run everything (leave the tab open)

In [ ]:
model, rows = core.sweep(model, tok, SCENARIOS, SETTINGS, n_test=100, out_csv=OUT)

## 3. Results

**How to read the recommendations:**

* *fewest examples: clearly beats prompt*: the smallest training set whose fine-tuned 95% range lies entirely above the prompt-only range. If this is **100 or less**, the default student setting (card B) will show a detectable change.
* *fewest examples: clearly beats prompt+docs*: the stronger test. Fine-tuning should beat pasting the policy into every prompt, too.
* **"not reached"** means no setting cleared the bar. Raise the default `N_EXAMPLES` (or `EPOCHS`) for that scenario, or don't assign it.

Copy the numbers into the *Dry-run results* table in `instructor/INSTRUCTOR-GUIDE.md`.

In [ ]:
table, recs = core.sweep_report(OUT)
display(recs)
display(table)

In [ ]:
# Download the raw results (one row per run, including per-field accuracy)
from google.colab import files
files.download(OUT)